# Chapter 12 — From Measurements to Policy

**Book alignment:** Hallucination From First Principles, Chapter 12

**Question this notebook isolates:** Does the reference policy engine return the documented commitment/action per case, with policy-v1 vs v2 replay changing only the provenance-unverified case?

All records below are deterministic synthetic fixtures replayed through the book's reference policy engine; they demonstrate the mechanism type, not real model behavior.

In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "experiments" / "hallucination-from-first-principles" / "policy_engine.py").exists():
            return candidate
    raise RuntimeError(
        "Run this notebook from a checkout containing experiments/hallucination-from-first-principles/policy_engine.py"
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
EXP_ROOT = REPO_ROOT / "experiments" / "hallucination-from-first-principles"
sys.path.insert(0, str(EXP_ROOT))

from policy_engine import Context, History, evaluate_policy

CTX = Context()
print("imported policy engine from:", EXP_ROOT)

## The documented cases return the claimed routes

Replay the Chapter 12 §11 cases verbatim: the engine must emit the commitment/action the chapter claims, including precedence (DENY keeps control while the losing rule stays in the trace).

In [ ]:
CLEAN = {"containment": "PASS", "relation_fidelity": "PASS",
         "epistemic_adequacy": "ANSWERABLE", "provenance": "VERIFIED"}

CASES = [
    ("clean", CLEAN, History(), ("PERMIT", "NONE")),
    ("provenance = UNVERIFIED", {**CLEAN, "provenance": "UNVERIFIED"}, History(), ("HOLD", "VERIFY")),
    ("policy_restricted + sensitivity = FAIL", {"policy_restricted": True, "sensitivity": "FAIL"},
     History(), ("DENY", "REFUSE_REQUEST")),
    ("recoverable evidence gap, first attempt", {"epistemic_adequacy": "RECOVERABLE_EVIDENCE_GAP"},
     History(retrieve_attempts=0), ("HOLD", "RETRIEVE")),
    ("recoverable evidence gap, retrieval budget exhausted",
     {"epistemic_adequacy": "RECOVERABLE_EVIDENCE_GAP"},
     History(retrieve_attempts=1), ("HOLD", "REVIEW")),
    ("user information gap, clarification budget exhausted", {"epistemic_adequacy": "USER_GAP"},
     History(ask_attempts=2), ("HOLD", "REVIEW")),
]

decisions = []
for name, rec, hist in [(c[0], c[1], c[2]) for c in CASES]:
    d = evaluate_policy(rec, CTX, hist)
    decisions.append((name, d))
    extra = f" matched={d['matched_rule']}" + (f" suppressed={d['suppressed_rules']}" if d["suppressed_rules"] else "")
    print(f"{name:52s} -> {d['commitment']} / {d['next_action']}{extra}")
assert decisions and decisions[0][1]["policy_version"] == "policy-v2"

In [ ]:
for (name, _rec, _hist, expected), (_n2, d) in zip(CASES, decisions):
    assert (d["commitment"], d["next_action"]) == expected, (name, d)
prohib = decisions[2][1]
assert prohib["matched_rule"] == "POLICY_PROHIBITED"
assert "SENSITIVITY_REVIEW" in prohib["suppressed_rules"]
assert "POLICY_PROHIBITED" in prohib["reason_codes"] and "SENSITIVITY_FAIL" in prohib["reason_codes"]
assert set(decisions[0][1]) >= {"commitment", "next_action", "matched_rule", "suppressed_rules",
    "reason_codes", "obligations", "step_index", "policy_version"}
print("PASS: all six documented cases match; DENY preserves the suppressed diagnosis")

## Policy-v1 vs v2 replay changes only the provenance-unverified case

Freeze candidates, measurements, and evidence; change only the policy (v1 omits the provenance rule and its clean-accept line, v2 restores both). Measurement is not authorization: nothing about the candidate moves, yet the verdict does.

In [ ]:
REPLAY = [
    ("clean", CLEAN, History()),
    ("provenance unverified", {**CLEAN, "provenance": "UNVERIFIED"}, History()),
    ("recoverable evidence gap", {"epistemic_adequacy": "RECOVERABLE_EVIDENCE_GAP"}, History()),
    ("sensitivity failure", {**CLEAN, "sensitivity": "FAIL"}, History()),
    ("prohibited request", {"policy_restricted": True}, History()),
]

rows = []
for name, rec, hist in REPLAY:
    v1 = evaluate_policy(rec, CTX, hist, require_provenance=False)
    v2 = evaluate_policy(rec, CTX, hist, require_provenance=True)
    s1 = (v1["commitment"], v1["next_action"])
    s2 = (v2["commitment"], v2["next_action"])
    rows.append((name, s1, s2))
    flag = "  <-- changed" if s1 != s2 else ""
    print(f"{name:26s} | v1 {s1[0]}/{s1[1]:14s} | v2 {s2[0]}/{s2[1]:14s}{flag}")

In [ ]:
changed = [name for name, s1, s2 in rows if s1 != s2]
assert changed == ["provenance unverified"], changed
prov = [r for r in rows if r[0] == "provenance unverified"][0]
assert prov[1] == ("PERMIT", "NONE") and prov[2] == ("HOLD", "VERIFY")
print("PASS: exactly one row changes (PERMIT/NONE -> HOLD/VERIFY); the flaw was policy, not measurement")

## Unknown is never permission; recovery always terminates

A critical field in an unknown state must not smuggle a PERMIT, and a recovery route with an exhausted budget must land on the terminal route instead of retrying forever.

In [ ]:
d_unknown = evaluate_policy({**CLEAN, "provenance": "NOT_MEASURED"}, CTX, History())
d_verify_spent = evaluate_policy({**CLEAN, "provenance": "UNVERIFIED"}, CTX,
                                   History(verify_attempts=1))
d_max_steps = evaluate_policy(CLEAN, CTX, History(step_index=4))

print("NOT_MEASURED provenance   ->", d_unknown["commitment"], "/", d_unknown["next_action"])
print("verify budget exhausted   ->", d_verify_spent["commitment"], "/", d_verify_spent["next_action"])
print("max steps exceeded        ->", d_max_steps["commitment"], "/", d_max_steps["next_action"],
      f"matched={d_max_steps['matched_rule']}")

In [ ]:
assert (d_unknown["commitment"], d_unknown["next_action"]) == ("HOLD", "VERIFY")
assert d_unknown["commitment"] != "PERMIT"
assert (d_verify_spent["commitment"], d_verify_spent["next_action"]) == ("HOLD", "REVIEW")
assert d_max_steps["matched_rule"] == "MAX_STEPS_EXCEEDED"
print("PASS: unknown blocks commit; exhausted budgets terminate instead of looping")

## What we earned

The typed reliability record authorizes nothing by itself; the versioned policy decides commitment and recovery, preserves losing diagnoses in the trace, bounds every retry with execution memory, and replays deterministically. Generation may be stochastic — acceptance is not.

Notebook 13 / Chapter 13 executes the routes policy only instructs: verification, repair, and rejection.